## RAG Pipelines - Data ingestion to Vector DB Pipeline

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from pathlib import Path

In [12]:
def load_documents(source):
    dir_loader = DirectoryLoader(
        path=source,
        glob="*.pdf",
        loader_cls=PyMuPDFLoader,
        show_progress=True
    )

    documents = dir_loader.load()
    print(f"{len(documents)} are loaded")
    return documents

In [13]:
documents = load_documents("/home/sonip/projects/agentic-ai/src/agentic_ai/updatedlangchain/rag/data")


100%|██████████| 2/2 [00:00<00:00,  2.24it/s]

506 are loaded


RecursiveCharacterTextSplitter splits long text into chunks of at most chunk_size characters, trying to keep semantically related text together.

It works by trying a list of separators in priority order, by default ["\n\n", "\n", " ", ""]:

1. Split on the first separator (paragraphs).
2. Merge the resulting pieces back together up to chunk_size.
3. If any piece is still too big, recurse on it with the next separator (lines, then words, then single characters).

So it prefers breaking at paragraph boundaries, then lines, then words, and only cuts mid-word as a last resort.

chunk_overlap repeats the tail of one chunk at the start of the next, so context isn't lost at the boundaries.

It's the usual default splitter for RAG pipelines because it keeps chunks close to the size limit without splitting sentences or paragraphs more than necessary.

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def make_chunks(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", " ", ""],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunked_documents = text_splitter.split_documents(documents)
    print(f"{len(documents)} are chunked into {len(chunked_documents)} using chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")
    return chunked_documents
    

In [26]:
chunked_documents = make_chunks(documents)

506 are chunked into 1632 using chunk_size=1000, chunk_overlap=200


In [27]:
chunked_documents[0]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': '/home/sonip/projects/agentic-ai/src/agentic_ai/updatedlangchain/rag/data/attention-is-all-you-need.pdf', 'file_path': '/home/sonip/projects/agentic-ai/src/agentic_ai/updatedlangchain/rag/data/attention-is-all-you-need.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz

## Embedding and VectorDB

### `sentence_transformers`
sentence-transformers is a Python library that turns text (sentences, paragraphs, chunks) into dense numeric vectors called embeddings. Texts with similar meaning get vectors that are close together.

Uses:
- Semantic search / RAG: embed your document chunks and the user's query, then retrieve the closest chunks from a vector store (FAISS, Chroma, etc.). This is its role in your RAG pipeline.
- Similarity and deduplication: compare how close two texts are in meaning, not just in shared words.
- Clustering and classification: group or label texts using their embeddings.
- Reranking: cross-encoder models score query and document pairs for more accurate ordering.

It runs locally and for free. Models like `all-MiniLM-L6-v2` are small and fast, so you don't need an API such as OpenAI embeddings. In LangChain you usually reach it through HuggingFaceEmbeddings, which wraps it.

In [36]:
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

In [34]:
# Define embedding model
class EmbeddingManager:
    """Documents embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


In [35]:
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1576.59it/s]


Model loaded successfully. Embedding dimension: 384


In [39]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vectore database"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """initialize chromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
        except Exception as e:
            print(f"Error initializing vector: {e}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and corresponding embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store ...")
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

            # Add to collection
            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )
            except Exception as e:
                print(f"Error adding documents to vector store: {e}")

vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents


In [41]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunked_documents]

embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 1632 texts...


Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Batches: 100%|██████████| 51/51 [04:04<00:00,  4.79s/it]

Generated embeddings with shape: (1632, 384)


In [42]:
vector_store.add_documents(chunked_documents, embeddings)

Adding 1632 documents to vector store ...
